# LLMOps Assignment 2 — Ollama `llama3.1:8b`
**Local inference | 5 questions × 3 prompt versions | MLflow | CSV exports**

No Gemini, Groq, OpenAI API, or API key is required. Ollama runs the model locally.
Local API token cost is recorded as zero; VM/GPU/CPU and electricity costs are excluded.

## Setup (run in VM terminal)

```bash
curl -fsSL https://ollama.com/install.sh | sh
ollama pull llama3.1:8b
# Ollama is often already running as a service. If not:
ollama serve

source venv/bin/activate
pip install -r requirements.txt
```

Keep Ollama running while executing this notebook. If Ollama runs on another host, set
`OLLAMA_HOST` to its reachable address before starting Jupyter.

In [ ]:
%pip install -q --upgrade ollama mlflow pandas numpy matplotlib

In [ ]:
import os, json, time, platform
from pathlib import Path
from datetime import datetime, timezone

import ollama
import pandas as pd
import numpy as np
import mlflow
from mlflow import MlflowClient

MODEL_NAME = "llama3.1:8b"
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
EXPERIMENT_NAME = "eduassist_llmops_assignment"
OUTPUT_DIR = Path("llmops_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

client = ollama.Client(host=OLLAMA_HOST)
print("Python:", platform.python_version())
print("Ollama host:", OLLAMA_HOST)
print("Model:", MODEL_NAME)

In [ ]:
# Verify Ollama connectivity and model availability
models = client.list().models
available = [m.model for m in models]
print("Available models:", available)
if not any(MODEL_NAME in name for name in available):
    raise RuntimeError(f"{MODEL_NAME} not found. Run: ollama pull {MODEL_NAME}")
print("Ollama is ready.")

## 1. Academic questions and prompt versions

In [ ]:
questions = [
    ("Q1", "What is machine learning, and how does it differ from traditional rule-based programming?"),
    ("Q2", "Explain supervised learning, unsupervised learning, and reinforcement learning with one example each."),
    ("Q3", "What is overfitting in machine learning? Explain two techniques used to reduce it."),
    ("Q4", "Describe the purpose of train, validation, and test datasets in a machine learning workflow."),
    ("Q5", "What is MLOps, and how do experiment tracking, CI/CD, and model monitoring support production machine learning?")
]

prompts = {
    "v1_basic": (
        "Answer this academic question clearly and accurately using simple language. "
        "Question: {question}"
    ),
    "v2_structured": (
        "You are a university academic tutor. Answer accurately using these headings: "
        "Definition, Explanation, Example, and Key takeaway. Be concise. Question: {question}"
    ),
    "v3_strict_json": (
        "Answer accurately. Return ONLY a valid JSON object with exactly these keys: "
        '"definition", "explanation", "example", "key_takeaway". '
        "Every value must be a string. No Markdown fences or extra text. Question: {question}"
    )
}
print("Total planned evaluations:", len(questions) * len(prompts))

## 2. Model call, retry handling, latency and token extraction

In [ ]:
def call_local_llm(prompt, json_mode=False, retries=2):
    last_error = None
    for attempt in range(retries + 1):
        started_at = datetime.now(timezone.utc).isoformat()
        start = time.perf_counter()
        try:
            args = {
                "model": MODEL_NAME,
                "messages": [{"role": "user", "content": prompt}],
                "options": {"temperature": 0.2}
            }
            if json_mode:
                args["format"] = "json"

            response = client.chat(**args)
            latency = time.perf_counter() - start
            answer = response.message.content or ""
            input_tokens = int(response.prompt_eval_count or 0)
            output_tokens = int(response.eval_count or 0)

            valid_json = None
            if json_mode:
                try:
                    obj = json.loads(answer)
                    valid_json = (
                        isinstance(obj, dict)
                        and set(obj) == {"definition", "explanation", "example", "key_takeaway"}
                        and all(isinstance(v, str) for v in obj.values())
                    )
                except (ValueError, TypeError):
                    valid_json = False

            return {
                "started_at": started_at,
                "ended_at": datetime.now(timezone.utc).isoformat(),
                "model": MODEL_NAME,
                "latency_seconds": latency,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "total_tokens": input_tokens + output_tokens,
                "cost_per_request": 0.0,
                "valid_json": valid_json,
                "status": "SUCCESS",
                "error": None,
                "response": answer
            }
        except Exception as exc:
            last_error = repr(exc)
            if attempt < retries:
                time.sleep(2 ** attempt)

    return {
        "started_at": None, "ended_at": datetime.now(timezone.utc).isoformat(),
        "model": MODEL_NAME, "latency_seconds": np.nan,
        "input_tokens": 0, "output_tokens": 0, "total_tokens": 0,
        "cost_per_request": 0.0, "valid_json": False if json_mode else None,
        "status": "FAILED", "error": last_error, "response": None
    }

## 3. Run 15 evaluations (5 questions × 3 prompt versions)

In [ ]:
results = []
for version, template in prompts.items():
    print("Running", version)
    for qid, question in questions:
        prompt = template.format(question=question)
        metrics = call_local_llm(prompt, json_mode=(version == "v3_strict_json"))
        results.append({
            "question_id": qid,
            "question": question,
            "prompt_version": version,
            **metrics
        })
        print(qid, metrics["status"], metrics["latency_seconds"], metrics["total_tokens"])
        time.sleep(0.5)

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR / "all_evaluation_results.csv", index=False)
display(results_df[[
    "question_id", "prompt_version", "model", "latency_seconds",
    "input_tokens", "output_tokens", "total_tokens", "valid_json", "status"
]])

## 4. Latency and token statistics

In [ ]:
successful = results_df[results_df.status == "SUCCESS"].copy()

def stats(series):
    x = pd.to_numeric(series, errors="coerce").dropna()
    return pd.Series({
        "mean": x.mean() if len(x) else np.nan,
        "median": x.median() if len(x) else np.nan,
        "min": x.min() if len(x) else np.nan,
        "max": x.max() if len(x) else np.nan,
        "std": x.std(ddof=1) if len(x) > 1 else (0.0 if len(x) else np.nan)
    })

overall_stats = pd.DataFrame({
    metric: stats(successful[metric])
    for metric in ["latency_seconds", "input_tokens", "output_tokens", "total_tokens"]
}).T
overall_stats.index.name = "metric"
overall_stats.to_csv(OUTPUT_DIR / "overall_statistics.csv")
display(overall_stats)

summary = successful.groupby("prompt_version").agg(
    successful_requests=("status", "count"),
    avg_latency_seconds=("latency_seconds", "mean"),
    median_latency_seconds=("latency_seconds", "median"),
    min_latency_seconds=("latency_seconds", "min"),
    max_latency_seconds=("latency_seconds", "max"),
    std_latency_seconds=("latency_seconds", "std"),
    avg_input_tokens=("input_tokens", "mean"),
    avg_output_tokens=("output_tokens", "mean"),
    avg_total_tokens=("total_tokens", "mean"),
    avg_cost_per_request=("cost_per_request", "mean")
).reset_index()

json_rate = successful.groupby("prompt_version")["valid_json"].apply(
    lambda s: s.dropna().astype(bool).mean() if s.notna().any() else np.nan
)
summary["valid_json"] = summary["prompt_version"].map(json_rate)
summary.to_csv(OUTPUT_DIR / "prompt_evaluation_summary.csv", index=False)
display(summary)

## 5. Cost and 10,000-request monthly projection

In [ ]:
# Local Ollama has no per-token API billing. Hardware/VM/electricity cost is excluded.
MONTHLY_REQUESTS = 10_000
costs = summary[[
    "prompt_version", "avg_input_tokens", "avg_output_tokens", "avg_cost_per_request"
]].copy()
costs["requests_per_month"] = MONTHLY_REQUESTS
costs["projected_monthly_api_cost"] = costs["avg_cost_per_request"] * MONTHLY_REQUESTS
costs["cost_note"] = "Zero API token charge; local compute/electricity costs excluded"
costs.to_csv(OUTPUT_DIR / "cost_and_monthly_projection.csv", index=False)
display(costs)

## 6. MLflow experiment tracking

In [ ]:
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "file:./mlruns"))
mlflow.set_experiment(EXPERIMENT_NAME)
mlflow_client = MlflowClient()
run_ids = {}

for version, template in prompts.items():
    group = successful[successful.prompt_version == version]
    j = group.valid_json.dropna()
    metrics = {
        "avg_latency_seconds": float(group.latency_seconds.mean()) if len(group) else 0.0,
        "avg_input_tokens": float(group.input_tokens.mean()) if len(group) else 0.0,
        "avg_output_tokens": float(group.output_tokens.mean()) if len(group) else 0.0,
        "avg_cost_per_request": 0.0,
        "valid_json": float(j.astype(bool).mean()) if len(j) else 0.0
    }

    with mlflow.start_run(run_name=version) as run:
        run_ids[version] = run.info.run_id
        mlflow.log_params({
            "provider": "Ollama local",
            "model_name": MODEL_NAME,
            "prompt_version": version,
            "question_count": len(questions),
            "temperature": 0.2
        })
        mlflow.log_metrics(metrics)

        prompt_path = OUTPUT_DIR / f"{version}_prompt.txt"
        prompt_path.write_text(template, encoding="utf-8")
        mlflow.log_artifact(str(prompt_path), artifact_path="prompts")

        version_csv = OUTPUT_DIR / f"{version}_results.csv"
        group.to_csv(version_csv, index=False)
        mlflow.log_artifact(str(version_csv), artifact_path="results")

print("MLflow experiment:", EXPERIMENT_NAME)
print("MLflow run IDs:", run_ids)

## 7. Select best prompt and register versions / production alias

In [ ]:
# Selection: highest JSON validity; ties broken by lowest mean latency.
ranked = summary.copy()
ranked["json_score"] = ranked.valid_json.fillna(0.0)
ranked["latency_sort"] = ranked.avg_latency_seconds.fillna(float("inf"))
ranked = ranked.sort_values(["json_score", "latency_sort"], ascending=[False, True])
best_prompt = ranked.iloc[0].prompt_version
print("Best prompt by measured rule:", best_prompt)

# MLflow Prompt Registry is version-dependent. Attempt registration and alias assignment.
registry_status = {"registered": [], "production_alias": None, "warning": None}
registry_name = "eduassist_academic_tutor"

try:
    import inspect
    register_prompt = getattr(getattr(mlflow, "genai", None), "register_prompt", None)
    if register_prompt is None:
        raise RuntimeError("This MLflow installation does not expose mlflow.genai.register_prompt.")

    signature = inspect.signature(register_prompt)
    for version, template in prompts.items():
        kwargs = {"name": registry_name, "template": template}
        if "commit_message" in signature.parameters:
            kwargs["commit_message"] = f"Assignment prompt {version}"
        elif "description" in signature.parameters:
            kwargs["description"] = f"Assignment prompt {version}"
        register_prompt(**kwargs)
        registry_status["registered"].append(version)

    # Identify the exact registered version by matching the prompt template if possible.
    versions = mlflow_client.search_prompt_versions(name=registry_name)
    chosen = None
    for item in versions:
        if getattr(item, "template", None) == prompts[best_prompt]:
            chosen = item
            break
    if chosen is None and versions:
        # Registration order follows prompt dict insertion order.
        chosen = sorted(versions, key=lambda x: int(x.version))[
            list(prompts.keys()).index(best_prompt)
        ]

    if chosen is None:
        raise RuntimeError("No prompt versions were returned after registration.")
    mlflow_client.set_prompt_alias(
        name=registry_name, alias="production", version=chosen.version
    )
    registry_status["production_alias"] = str(chosen.version)
    print("Production alias assigned to prompt version:", chosen.version)

except Exception as exc:
    registry_status["warning"] = str(exc)
    print("Prompt Registry warning:", exc)
    print("Prompt templates and metrics remain logged as MLflow artifacts and runs.")
    print("If required by your course, upgrade MLflow and verify Prompt Registry API compatibility.")

(OUTPUT_DIR / "prompt_registry_status.json").write_text(
    json.dumps(registry_status, indent=2), encoding="utf-8"
)

## 8. Final exports and execution summary

In [ ]:
export_cols = [
    "question_id", "prompt_version", "model", "started_at", "ended_at",
    "latency_seconds", "input_tokens", "output_tokens", "total_tokens",
    "cost_per_request", "valid_json", "status", "error"
]
results_df[export_cols].to_csv(OUTPUT_DIR / "latency_tokens_cost.csv", index=False)
summary.to_csv(OUTPUT_DIR / "prompt_metrics.csv", index=False)

execution_summary = {
    "provider": "Ollama local",
    "model": MODEL_NAME,
    "experiment_name": EXPERIMENT_NAME,
    "question_count": len(questions),
    "prompt_versions": list(prompts),
    "total_evaluations": len(results_df),
    "successful_evaluations": int((results_df.status == "SUCCESS").sum()),
    "failed_evaluations": int((results_df.status != "SUCCESS").sum()),
    "selected_best_prompt": best_prompt,
    "actual_api_cost": 0.0,
    "monthly_requests_projection": MONTHLY_REQUESTS,
    "projected_monthly_api_cost": 0.0,
    "compute_cost_included": False,
    "mlflow_run_ids": run_ids,
    "prompt_registry_status": registry_status
}
(OUTPUT_DIR / "execution_summary.json").write_text(
    json.dumps(execution_summary, indent=2), encoding="utf-8"
)
print(json.dumps(execution_summary, indent=2))
print("All outputs:", OUTPUT_DIR.resolve())

## 9. Optional charts

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.bar(summary.prompt_version, summary.avg_latency_seconds)
plt.title("Average Latency by Prompt Version")
plt.xlabel("Prompt Version")
plt.ylabel("Seconds")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "average_latency_by_prompt.png", dpi=150)
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(summary.prompt_version, summary.avg_total_tokens)
plt.title("Average Total Tokens by Prompt Version")
plt.xlabel("Prompt Version")
plt.ylabel("Tokens")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "average_tokens_by_prompt.png", dpi=150)
plt.show()

## 10. Submission validation checklist

- [ ] All 15 runs executed; explain any failed requests.
- [ ] Request-level CSV contains timestamps, latency, and token counts.
- [ ] Prompt-level CSV contains all five required MLflow metrics.
- [ ] MLflow experiment `eduassist_llmops_assignment` contains three prompt-version runs.
- [ ] Capture MLflow UI screenshots of the experiment and runs.
- [ ] Verify prompt registry versions and `production` alias; document any API compatibility warning.
- [ ] Report exact Ollama/model version, VM CPU/RAM/GPU, Python and MLflow versions, and execution date.
- [ ] State clearly that API cost is zero for local Ollama, while compute/electricity costs are excluded.